In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge
### Deep Learning & Generative AI Project 

This notebook walks through the complete pipeline for the Smart MCQ Solver
Challenge — from data loading and EDA, through a simple baseline, up to the
three main models built for this project, an additional experiment, and a
final comparison of all approaches.

**Task:** For each question (prompt + 5 options A–E), predict the top-3 most
likely correct answers in ranked order. Evaluated using **MAP@3**.

**Notebook Structure:**
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Visualizations
4. Baseline Model — TF-IDF + Cosine Similarity
5. TF-IDF + Logistic Regression
6. My 3 Main Models (From-Scratch BiLSTM+Attention, Logistic Regression, ELECTRA)
7. Additional Experiment — Sentence-Transformers (MPNet)
8. Model Comparison


#  **Setup & Imports**

In [2]:
!pip install wandb -q 

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style("whitegrid")


# **Load Data**

In [3]:
DATA_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTION_COLS = ["A", "B", "C", "D", "E"]


In [4]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head(5)


Train shape: (2000, 8)
Test shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# **Exploratory Data Analysis**

## Basic Info

In [5]:
print("TRAIN INFO:")
print(train.info())


TRAIN INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None


## Missing Values

In [6]:
print("MISSING VALUES")
print(train.isnull().sum())


MISSING VALUES
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


##  Duplicate Values

In [7]:
print("DUPLICATES:")
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate prompts      :", train['prompt'].duplicated().sum())


DUPLICATES:
Duplicate rows in train: 0
Duplicate prompts      : 242


In [8]:
# Duplicate prompts explore
dup_prompts = train[train['prompt'].duplicated(keep=False)].sort_values('prompt')
print(f"Total rows with duplicate prompts: {len(dup_prompts)}")
print(f"\nSame prompt, different answers example:")
print(dup_prompts[['prompt','answer']].head(6).to_string())

# same prompt have same ans or different?
same_answer = train.groupby('prompt')['answer'].nunique()
print(f"\nPrompts with same answer always  : {(same_answer == 1).sum()}")
print(f"Prompts with different answers   : {(same_answer > 1).sum()}")


Total rows with duplicate prompts: 454

Same prompt, different answers example:
                                                                                                         prompt answer
605                                           Choose the correct answer: What are permutation-inversion groups?      E
456                                           Choose the correct answer: What are permutation-inversion groups?      E
439                                           Choose the correct answer: What are permutation-inversion groups?      E
1464              Choose the correct answer: What are permutation-inversion groups? based on the given context.      E
42                Choose the correct answer: What are permutation-inversion groups? based on the given context.      E
484   Choose the correct answer: What are the four qualitative levels of crystallinity described by geologists?      B

Prompts with same answer always  : 1758
Prompts with different answers   : 0


##  Text Cleaning

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

train['cleaned_prompt'] = train['prompt'].apply(clean_text)
test['cleaned_prompt']  = test['prompt'].apply(clean_text)

for opt in OPTION_COLS:
    train[f'cleaned_{opt}'] = train[opt].apply(clean_text)

print("Text cleaning done.")
print("\nSample cleaned prompt:")
print(train['cleaned_prompt'].iloc[0])


Text cleaning done.

Sample cleaned prompt:
pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options


##  Prompt Length Analysis

In [10]:
train['prompt_word_count'] = train['cleaned_prompt'].apply(lambda x: len(x.split()))
train['prompt_char_count'] = train['cleaned_prompt'].apply(len)
test['prompt_word_count']  = test['cleaned_prompt'].apply(lambda x: len(x.split()))

print("Train Prompt Word Count:")
print(train['prompt_word_count'].describe().round(2))


Train Prompt Word Count:
count    2000.00
mean       18.15
std         6.78
min         3.00
25%        14.00
50%        17.00
75%        22.00
max        51.00
Name: prompt_word_count, dtype: float64


## Option Length Comparison

In [11]:
option_lengths = {}
for opt in OPTION_COLS:
    option_lengths[opt] = train[f'cleaned_{opt}'].apply(lambda x: len(x.split()))

print("Average word count per option:")
for opt in OPTION_COLS:
    print(f"  Option {opt}: {option_lengths[opt].mean():.2f}")


Average word count per option:
  Option A: 26.12
  Option B: 26.50
  Option C: 26.52
  Option D: 25.97
  Option E: 26.16


## Correct Option Length vs Wrong Option Length

In [12]:
correct_lengths = []
wrong_lengths   = []

for _, row in train.iterrows():
    for opt in OPTION_COLS:
        length = len(str(row[opt]).split())
        if opt == row['answer']:
            correct_lengths.append(length)
        else:
            wrong_lengths.append(length)

print(f"Avg length of correct options: {np.mean(correct_lengths):.2f}")
print(f"Avg length of wrong options  : {np.mean(wrong_lengths):.2f}")


Avg length of correct options: 28.66
Avg length of wrong options  : 25.68


## Vocabulary Size 

In [13]:
all_text = ' '.join(train['cleaned_prompt'].tolist())
all_prompt_words = set(all_text.split())

all_option_words = set()
for opt in OPTION_COLS:
    opt_text = ' '.join(train[f'cleaned_{opt}'].tolist())
    all_option_words.update(opt_text.split())

print(f"Vocab size (prompts only)  : {len(all_prompt_words)}")
print(f"Vocab size (options only)  : {len(all_option_words)}")
print(f"Combined vocab size        : {len(all_prompt_words | all_option_words)}")
print(f"Overlap (prompt & options) : {len(all_prompt_words & all_option_words)}")


Vocab size (prompts only)  : 859
Vocab size (options only)  : 2885
Combined vocab size        : 3096
Overlap (prompt & options) : 648


## Sample Questions

In [14]:
print("=" * 65)
for i in range(2):
    row = train.iloc[i]
    print(f"ID     : {row['id']}")
    print(f"Prompt : {row['prompt']}")
    for opt in OPTION_COLS:
        marker = "  <-- CORRECT" if opt == row['answer'] else ""
        print(f"  {opt}: {row[opt]}{marker}")
    print("=" * 65)


ID     : 1
Prompt : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
  A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
  B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.  <-- CORRECT
  C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.
  D: Martin Heidegg